In [4]:
import pandas as pd

CALC_TYPES = [
    "mbar",
    "dg-c2-pb",
    "dg-c2-gb",
    "dg-ie-pb",
    "dg-ie-gb",
    "dh-pb",
    "dh-gb",
]


# Define the custom orders
SYSTEM_ORDER = ["p38", "A2A", "ptp1b", "tyk2", "thrombin", "mcl1", "CyclophilinD", "SAMPL6-OA"]
FORCE_FIELD_ORDER = ['espaloma-0.3.1', 'gaff-2.11', 'openff-2.0.0']

SYSTEM_NAME = {
    "p38": "P38", 
    "A2A": "A2A",
    "ptp1b": "PTP1B",
    "tyk2": "TYK2",
    "thrombin": "Thrombin",
    "mcl1": "MCL1",
    "CyclophilinD": "CyclophilinD",
    "SAMPL6-OA": "SAMPL6-OA"
}


BindFlowData = pd.read_csv("../BindFlow.csv", index_col=0)

columns = [
    "system",
    "ligand",
    "replica",
    "sample",
    "exp_dG",
    "exp_dG_error",
]
for CALC_TYPE in CALC_TYPES:
    columns += [
    f"simulation_{CALC_TYPE}_espaloma-0.3.1",
    f"simulation_{CALC_TYPE}_gaff-2.11",
    f"simulation_{CALC_TYPE}_openff-2.0.0",
    ]
BindFlowData = BindFlowData[columns]

BindFlowData.rename(
    columns={
        "system": "source",
    },
    inplace=True
)

In [5]:
mean = BindFlowData.groupby(["source", "ligand"]).mean().reset_index().drop(columns=["replica", "sample"])
sem = BindFlowData.groupby(["source", "ligand"]).sem().reset_index().drop(columns=["replica", "sample"])


# Filter DataFrame
mask = mean["simulation_mbar_espaloma-0.3.1"].notna()
mean = mean[mask]
sem = sem[mask]

sem.rename(
    columns={column: column.replace("simulation_", "sem_") for column in BindFlowData.columns},
    inplace=True
)
sem.drop(columns=["exp_dG", "exp_dG_error"], inplace=True)
df_merge = pd.merge(mean, sem, on=["source", "ligand"])

In [6]:
(df_merge[(df_merge["source"] == "ptp1b") & df_merge["simulation_mbar_espaloma-0.3.1"].notna()]["sem_mbar_gaff-2.11"])

81     3.993684
82     0.860103
83     1.798093
84     0.864688
85     1.332339
86     2.433706
87     1.078678
88     1.520327
89     2.030807
90     0.674146
91     2.032155
92     2.137923
93     0.589296
94     2.679208
95     2.183325
96     1.893684
97     0.584323
98     1.286703
99     2.277754
100    1.225726
101    1.261226
102    1.624243
Name: sem_mbar_gaff-2.11, dtype: float64